# Polymarket Sportsbook Arbitrage Feasibility

    This notebook is a research-only replication of the core idea in the Kacho article: compare sportsbook odds against live Polymarket orderbooks and look for places where Polymarket asks sit below a sportsbook-derived fair probability.

    Phase 1 is deliberately bounded. It does not place orders, manage private keys, run continuously, or claim executable arbitrage. It builds a snapshot pipeline, shows the data, scores candidate opportunities, and records why markets were excluded.

## Strategy Recap

    The strategy has four moving pieces:

    1. Capture sportsbook decimal odds for head-to-head events.
    2. Convert odds to implied probabilities and remove two-way overround so the two sides sum to 1.
    3. Match sportsbook teams/events to Polymarket binary outcome tokens.
    4. Compare sportsbook fair probability to Polymarket best ask.

    The basic score is:

    ```text
    implied_prob = 1 / decimal_odds
    fair_prob = implied_prob / sum(two_way_implied_probs)
    gross_edge = sportsbook_fair_prob - polymarket_best_ask
    net_edge = gross_edge - slippage_buffer - fee_buffer
    ```

    A `taker_candidate` means the current Polymarket ask is below the modeled fair value after buffers. A `maker_candidate` means the spread is wide enough that a hypothetical maker quote could have edge; it is not treated as executable profit.

## Caveats

    OddsPortal HTML is a fragile source. The parser below supports simple page structures and saved fixtures; if live scraping is blocked or the layout changes, the notebook falls back to saved HTML rather than trying to bypass anti-bot controls.

    The probability model is intentionally simple: two-way overround removal only. It excludes 3-way/draw markets, spreads, totals, futures, parlays, and unclear mappings. Polymarket orderbooks are snapshots, so apparent edge can disappear before execution. Matching is fuzzy and should be manually reviewed before any real trading research.

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import httpx
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = Path("/Users/qisongqiao/Warehouse/cv/project_simulation/prediction_market")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from prediction_market.arbitrage import (
    capture_oddsportal_snapshots,
    fetch_polymarket_books_for_markets,
    fetch_polymarket_candidate_markets,
    load_oddsportal_html_files,
    match_odds_to_polymarket,
    parse_oddsportal_html,
    score_opportunities,
    write_arbitrage_tables,
)
from prediction_market.utils import compact_json, utc_now_iso

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 140)

RAW_ODDSPORTAL_DIR = PROJECT_ROOT / "data" / "arbitrage" / "raw" / "oddsportal"
RAW_POLYMARKET_DIR = PROJECT_ROOT / "data" / "arbitrage" / "raw" / "polymarket"
PROCESSED_DIR = PROJECT_ROOT / "data" / "arbitrage" / "processed"
for path in [RAW_ODDSPORTAL_DIR, RAW_POLYMARKET_DIR, PROCESSED_DIR]:
    path.mkdir(parents=True, exist_ok=True)

RUN_LIVE_ODDSPORTAL_CAPTURE = os.getenv("RUN_LIVE_ODDSPORTAL_CAPTURE") == "1"
RUN_LIVE_POLYMARKET_CAPTURE = os.getenv("RUN_LIVE_POLYMARKET_CAPTURE") == "1"

SLIPPAGE_BUFFER = float(os.getenv("ARBITRAGE_SLIPPAGE_BUFFER", "0.01"))
FEE_BUFFER = float(os.getenv("ARBITRAGE_FEE_BUFFER", "0.00"))
MIN_NET_EDGE = float(os.getenv("ARBITRAGE_MIN_NET_EDGE", "0.03"))
MIN_MATCH_SCORE = float(os.getenv("ARBITRAGE_MIN_MATCH_SCORE", "72"))

print(f"Project root: {PROJECT_ROOT}")
print(f"Live OddsPortal capture: {RUN_LIVE_ODDSPORTAL_CAPTURE}")
print(f"Live Polymarket capture: {RUN_LIVE_POLYMARKET_CAPTURE}")

## OddsPortal Snapshot

    By default this notebook parses a saved HTML fixture. To attempt a live manual snapshot, run Jupyter with `RUN_LIVE_ODDSPORTAL_CAPTURE=1` and install the Playwright browser first:

    ```bash
    .venv/bin/python -m playwright install chromium
    RUN_LIVE_ODDSPORTAL_CAPTURE=1 jupyter notebook notebooks/04_polymarket_bookmaker_arbitrage_feasibility.ipynb
    ```

    Live capture is best-effort only. If it fails, use saved HTML snapshots in `data/arbitrage/raw/oddsportal/`.

In [ ]:
ODDSPORTAL_URLS = [
    "https://www.oddsportal.com/esports/",
    "https://www.oddsportal.com/basketball/",
    "https://www.oddsportal.com/tennis/",
    "https://www.oddsportal.com/soccer/",
]

fixture_html = PROJECT_ROOT / "tests" / "fixtures" / "oddsportal_sample.html"
html_paths = []

if RUN_LIVE_ODDSPORTAL_CAPTURE:
    try:
        html_paths = capture_oddsportal_snapshots(ODDSPORTAL_URLS, RAW_ODDSPORTAL_DIR)
    except Exception as exc:
        print(f"Live OddsPortal capture failed; using fixture instead: {exc}")
        html_paths = [fixture_html]
else:
    html_paths = [fixture_html]

odds_quotes = load_oddsportal_html_files(html_paths)
odds_quotes.to_parquet(PROCESSED_DIR / "odds_quotes.parquet", index=False)
odds_quotes.to_csv(PROCESSED_DIR / "odds_quotes.csv", index=False)

odds_quotes[[
    "sport", "league", "event_id", "event_name", "team", "opponent",
    "decimal_odds", "fair_probability", "market_type", "exclusion_reason"
]]

### Raw OddsPortal Sample

    The raw source is saved as HTML. The parser extracts event/team names, decimal odds, bookmaker, start time, sport, league, and a compact raw payload for auditability.

In [ ]:
sample_path = Path(html_paths[0])
raw_html_excerpt = sample_path.read_text(encoding="utf-8")[:1200]
print(raw_html_excerpt)

## Polymarket Snapshot

    By default this notebook uses a tiny synthetic Polymarket-like orderbook fixture so the model can run without network calls. To query current Gamma markets and CLOB books, run with:

    ```bash
    RUN_LIVE_POLYMARKET_CAPTURE=1 jupyter notebook notebooks/04_polymarket_bookmaker_arbitrage_feasibility.ipynb
    ```

    Live capture writes raw JSON into `data/arbitrage/raw/polymarket/` and processed books into `data/arbitrage/processed/polymarket_books.parquet`.

In [ ]:
retrieved_at = utc_now_iso()

sample_market = {
    "id": "sample-liquid-fnatic",
    "conditionId": "0xsample",
    "slug": "team-liquid-vs-fnatic-sample",
    "question": "Will Team Liquid beat Fnatic?",
    "outcomes": ["Team Liquid", "Fnatic"],
    "clobTokenIds": ["sample-liquid", "sample-fnatic"],
    "events": [{"title": "Team Liquid vs Fnatic", "sport": "esports", "league": "League of Legends"}],
}
sample_orderbooks = {
    "sample-liquid": {"bids": [{"price": "0.47", "size": "100"}], "asks": [{"price": "0.49", "size": "120"}]},
    "sample-fnatic": {"bids": [{"price": "0.49", "size": "80"}], "asks": [{"price": "0.52", "size": "90"}]},
}

sample_polymarket_books = pd.DataFrame(
    [
        {
            "market_id": "sample-liquid-fnatic",
            "condition_id": "0xsample",
            "slug": "team-liquid-vs-fnatic-sample",
            "question": "Will Team Liquid beat Fnatic?",
            "event_title": "Team Liquid vs Fnatic",
            "sport": "esports",
            "league": "League of Legends",
            "outcome": "Team Liquid",
            "outcome_index": 0,
            "token_id": "sample-liquid",
            "best_bid": 0.47,
            "best_ask": 0.49,
            "bid_depth": 100.0,
            "ask_depth": 120.0,
            "spread": 0.02,
            "retrieved_at": retrieved_at,
            "exclusion_reason": "",
            "raw_market": compact_json(sample_market),
            "raw_orderbook": compact_json(sample_orderbooks["sample-liquid"]),
        },
        {
            "market_id": "sample-liquid-fnatic",
            "condition_id": "0xsample",
            "slug": "team-liquid-vs-fnatic-sample",
            "question": "Will Team Liquid beat Fnatic?",
            "event_title": "Team Liquid vs Fnatic",
            "sport": "esports",
            "league": "League of Legends",
            "outcome": "Fnatic",
            "outcome_index": 1,
            "token_id": "sample-fnatic",
            "best_bid": 0.49,
            "best_ask": 0.52,
            "bid_depth": 80.0,
            "ask_depth": 90.0,
            "spread": 0.03,
            "retrieved_at": retrieved_at,
            "exclusion_reason": "",
            "raw_market": compact_json(sample_market),
            "raw_orderbook": compact_json(sample_orderbooks["sample-fnatic"]),
        },
    ]
)

if RUN_LIVE_POLYMARKET_CAPTURE:
    with httpx.Client(timeout=30.0) as client:
        polymarket_markets = fetch_polymarket_candidate_markets(client, limit=200, include_excluded=True)
        (RAW_POLYMARKET_DIR / "gamma_markets.json").write_text(json.dumps(polymarket_markets, indent=2), encoding="utf-8")
        polymarket_books = fetch_polymarket_books_for_markets(client, polymarket_markets)
        (RAW_POLYMARKET_DIR / "polymarket_books_raw_rows.json").write_text(
            polymarket_books.to_json(orient="records", indent=2), encoding="utf-8"
        )
else:
    (RAW_POLYMARKET_DIR / "sample_gamma_markets.json").write_text(json.dumps([sample_market], indent=2), encoding="utf-8")
    (RAW_POLYMARKET_DIR / "sample_orderbooks.json").write_text(json.dumps(sample_orderbooks, indent=2), encoding="utf-8")
    polymarket_books = sample_polymarket_books

polymarket_books.to_parquet(PROCESSED_DIR / "polymarket_books.parquet", index=False)
polymarket_books.to_csv(PROCESSED_DIR / "polymarket_books.csv", index=False)

polymarket_books[[
    "market_id", "question", "outcome", "best_bid", "best_ask", "bid_depth", "ask_depth", "spread", "exclusion_reason"
]]

### Raw Polymarket Sample

    Polymarket outcome prices are outcome-token prices. A best ask of `0.49` means the token can currently be bought around 49 cents, ignoring fill probability, fees, market impact, and stale-data risk.

In [ ]:
raw_market_sample = json.loads(polymarket_books.iloc[0]["raw_market"])
raw_book_sample = json.loads(polymarket_books.iloc[0]["raw_orderbook"])
print(json.dumps({"market": raw_market_sample, "orderbook": raw_book_sample}, indent=2)[:1800])

## Market Filtering Diagnostics

    The first phase scores only binary winner/head-to-head style markets. Rows can be excluded because they are 3-way/draw markets, spreads/totals, missing CLOB tokens, missing asks, or low-confidence matches.

In [ ]:
odds_filter_summary = (
    odds_quotes.groupby(["market_type", "exclusion_reason"], dropna=False)
    .size()
    .reset_index(name="rows")
    .sort_values("rows", ascending=False)
)
poly_filter_summary = (
    polymarket_books.groupby("exclusion_reason", dropna=False)
    .size()
    .reset_index(name="rows")
    .sort_values("rows", ascending=False)
)

display(odds_filter_summary)
display(poly_filter_summary)

## Match Sportsbook Quotes To Polymarket Outcomes

    Matching combines three fuzzy signals:

    - Team/outcome name similarity.
    - Event title/question similarity.
    - League/title similarity when available.

    This is intentionally conservative. Live sports markets with `Yes`/`No` outcomes may need extra semantic parsing because the team name can live in the question rather than the outcome label.

In [ ]:
matched_events = match_odds_to_polymarket(
    odds_quotes,
    polymarket_books,
    min_score=MIN_MATCH_SCORE,
)
matched_events.to_parquet(PROCESSED_DIR / "matched_events.parquet", index=False)
matched_events.to_csv(PROCESSED_DIR / "matched_events.csv", index=False)

matched_events[[
    "source_event_name", "source_team", "fair_probability", "polymarket_question",
    "polymarket_outcome", "polymarket_best_bid", "polymarket_best_ask",
    "match_score", "event_score", "team_score", "league_score"
]]

In [ ]:
if matched_events.empty:
    print("No matches above threshold. Lower MIN_MATCH_SCORE for diagnostics, or inspect unmatched teams/questions manually.")
else:
    display(matched_events["match_score"].describe().to_frame("match_score"))
    display(
        matched_events.sort_values("match_score", ascending=True)[[
            "source_event_name", "source_team", "polymarket_question", "polymarket_outcome", "match_score"
        ]]
    )

## Opportunity Scoring

    The opportunity table includes both executable-looking taker candidates and non-executable maker candidates. Excluded rows stay in the output so the report is auditable.

In [ ]:
arbitrage_opportunities = score_opportunities(
    matched_events,
    slippage_buffer=SLIPPAGE_BUFFER,
    fee_buffer=FEE_BUFFER,
    min_net_edge=MIN_NET_EDGE,
    min_match_score=MIN_MATCH_SCORE,
)
arbitrage_opportunities.to_parquet(PROCESSED_DIR / "arbitrage_opportunities.parquet", index=False)
arbitrage_opportunities.to_csv(PROCESSED_DIR / "arbitrage_opportunities.csv", index=False)

arbitrage_opportunities[[
    "opportunity_type", "source_event_name", "source_team", "sportsbook",
    "fair_probability", "polymarket_best_bid", "polymarket_best_ask",
    "gross_edge", "net_edge", "maker_quote", "maker_net_edge", "match_score", "exclusion_reason"
]]

## Sensitivity

    This grid answers a practical question: does the snapshot still look interesting after changing edge thresholds and slippage assumptions?

In [ ]:
sensitivity_rows = []
for slippage in [0.00, 0.005, 0.01, 0.02]:
    for threshold in [0.01, 0.02, 0.03, 0.05]:
        scored = score_opportunities(
            matched_events,
            slippage_buffer=slippage,
            fee_buffer=FEE_BUFFER,
            min_net_edge=threshold,
            min_match_score=MIN_MATCH_SCORE,
        )
        sensitivity_rows.append(
            {
                "slippage_buffer": slippage,
                "min_net_edge": threshold,
                "taker_candidates": int((scored["opportunity_type"] == "taker_candidate").sum()) if not scored.empty else 0,
                "maker_candidates": int((scored["opportunity_type"] == "maker_candidate").sum()) if not scored.empty else 0,
                "max_net_edge": scored["net_edge"].max() if not scored.empty else None,
                "max_maker_net_edge": scored["maker_net_edge"].max() if not scored.empty else None,
            }
        )

sensitivity = pd.DataFrame(sensitivity_rows)
sensitivity.to_parquet(PROCESSED_DIR / "arbitrage_sensitivity.parquet", index=False)
sensitivity.to_csv(PROCESSED_DIR / "arbitrage_sensitivity.csv", index=False)
sensitivity

## Write Report Tables

    The four core outputs are written as Parquet and CSV:

    - `odds_quotes`: sportsbook odds by event/team/book/time.
    - `polymarket_books`: Polymarket candidate outcomes with best bid/ask/depth.
    - `matched_events`: sportsbook-to-Polymarket links with confidence scores.
    - `arbitrage_opportunities`: ranked edge candidates and exclusions.

In [ ]:
written = write_arbitrage_tables(
    {
        "odds_quotes": odds_quotes,
        "polymarket_books": polymarket_books,
        "matched_events": matched_events,
        "arbitrage_opportunities": arbitrage_opportunities,
    },
    PROCESSED_DIR,
)
written

## Feasibility Verdict

    The verdict is snapshot-specific:

    - `viable`: at least one taker candidate survives the configured buffers.
    - `weak`: no taker candidate, but at least one maker candidate appears interesting.
    - `not viable`: no candidate clears thresholds.

    For live research, a viable snapshot still needs manual verification of matching, freshness, available size, Polymarket fees/rewards, execution risk, and sportsbook withdrawal/limit constraints.

In [ ]:
taker_count = int((arbitrage_opportunities["opportunity_type"] == "taker_candidate").sum()) if not arbitrage_opportunities.empty else 0
maker_count = int((arbitrage_opportunities["opportunity_type"] == "maker_candidate").sum()) if not arbitrage_opportunities.empty else 0
max_net_edge = arbitrage_opportunities["net_edge"].max() if not arbitrage_opportunities.empty else None

if taker_count:
    verdict = "viable"
    reason = f"{taker_count} taker candidate(s) survived the configured buffers."
elif maker_count:
    verdict = "weak"
    reason = f"{maker_count} maker candidate(s) appeared, but no executable taker candidate cleared the threshold."
else:
    verdict = "not viable"
    reason = "No candidate cleared the configured thresholds in this snapshot."

verdict_summary = pd.DataFrame(
    [
        {
            "verdict": verdict,
            "reason": reason,
            "taker_candidates": taker_count,
            "maker_candidates": maker_count,
            "max_net_edge": max_net_edge,
            "slippage_buffer": SLIPPAGE_BUFFER,
            "fee_buffer": FEE_BUFFER,
            "min_net_edge": MIN_NET_EDGE,
            "min_match_score": MIN_MATCH_SCORE,
        }
    ]
)
verdict_summary.to_csv(PROCESSED_DIR / "arbitrage_verdict.csv", index=False)
verdict_summary